# 9장. 회귀 분석으로 숫자 예측하기

이 Notebook의 핵심은 높은 점수를 만드는 것이 아니라 다음 순서를 지키는 것입니다.

`예측 시점 → 누수 방지 → 시간 분할 → Train TimeSeriesSplit → 모델 선택 고정 → Final Test → Baseline 비교`

**Final Test 결과를 보고 모델을 고르지 않습니다.**

In [19]:
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path for path in (current, *current.parents)
        if (path / "scripts" / "prepare_ch09_data.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("scripts 폴더가 있는 프로젝트 루트를 찾지 못했습니다.")

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

for script in [
    "scripts/prepare_ch09_data.py",
    "scripts/run_regression_analysis.py",
]:
    print(f"\n실행: python {script}")
    result = subprocess.run(
        [sys.executable, script],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env,
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"{script} 실패 (return code: {result.returncode})")

validation_path = PROJECT_ROOT / "reports" / "ch09_regression_validation.csv"
validation = pd.read_csv(validation_path)

required_checks = {
    "forbidden_feature_overlap",
    "strict_train_before_test",
    "selected_model_exists_in_train_cv",
    "final_test_contains_baseline",
    "final_test_contains_frozen_selected_model",
    "test_rows_for_r2",
}
assert required_checks.issubset(set(validation["check"]))
assert validation["status"].eq("PASS").all(), validation

print("\n두 스크립트 실행 및 Validation 6개 항목: PASS")
display(validation)


실행: python scripts/prepare_ch09_data.py
Chapter09 모델링 입력 준비 완료
입력 Raw 경로: C:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\raw

[관계 검증]
                                               check  invalid_count
  orders.customer_id exists in customers.customer_id              0
      order_items.order_id exists in orders.order_id              0
order_items.product_id exists in products.product_id              0

[저장된 processed 파일]
- C:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\processed\customers_clean.csv
- C:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\processed\products_clean.csv
- C:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\processed\orders_clean.csv
- C:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\processed\order_items_clean.csv


실행: python scripts/run_regression_analysis.py
9장 회귀 분석 완료

[시간 순서 분할]
split  rows  ratio_pct start_date   end_date
train   244      81.33 2025-07-09 2026-04-13
 test    56 

,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [20]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

PROJECT_ROOT: C:\Users\ojh23\Documents\projects\llm-data-analysis-course
PROCESSED_DIR: C:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\processed


## 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

In [21]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

Target: order_total
Allowed features: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']
Forbidden features: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'product_id', 'quantity', 'total_quantity', 'unit_price']
Feature leakage gate: PASS


## 2. 검증된 전처리 데이터를 불러옵니다

In [22]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

customers (150, 6)
orders (300, 7)
order_items (764, 6)


## 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [23]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
display(model_data.head())
print('Model rows:', len(model_data))

,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek,order_total,gender,age,city
0,238,124,2025-07-09,bank_transfer,completed,7,2,361000,M,67,대구
1,79,62,2025-07-11,bank_transfer,refunded,7,4,283000,F,63,광주
2,12,98,2025-07-12,kakao_pay,completed,7,5,444000,F,61,인천
3,169,9,2025-07-12,kakao_pay,completed,7,5,448000,M,69,서울
4,152,62,2025-07-13,naver_pay,refunded,7,6,985000,F,63,광주


Model rows: 300


In [24]:
import numpy as np
import pandas as pd

items = data["order_items"].copy()
orders = data["orders"].copy()
customers = data["customers"].copy()

# 1) line_total = quantity × unit_price
quantity = pd.to_numeric(items["quantity"], errors="coerce")
unit_price = pd.to_numeric(items["unit_price"], errors="coerce")
line_total = pd.to_numeric(items["line_total"], errors="coerce")
calculated_line_total = quantity * unit_price

line_ok = np.isclose(
    line_total.to_numpy(dtype=float),
    calculated_line_total.to_numpy(dtype=float),
    rtol=1e-9,
    atol=1e-6,
)
line_mismatch_count = int((~line_ok).sum())

# 2) 주문별 상세 합계와 model_data의 order_total 비교
expected_target = (
    pd.DataFrame({
        "order_id": items["order_id"],
        "calculated_line_total": calculated_line_total,
    })
    .groupby("order_id", as_index=False)["calculated_line_total"]
    .sum(min_count=1)
)

target_compare = expected_target.merge(
    model_data[["order_id", "order_total"]],
    on="order_id",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

target_equal = np.isclose(
    pd.to_numeric(target_compare["calculated_line_total"], errors="coerce")
        .to_numpy(dtype=float),
    pd.to_numeric(target_compare["order_total"], errors="coerce")
        .to_numpy(dtype=float),
    rtol=1e-9,
    atol=1e-6,
)
target_ok = target_compare["_merge"].eq("both").to_numpy() & target_equal
target_mismatch_count = int((~target_ok).sum())

# 3) orders ↔ order_total: one_to_one
orders_target = orders[["order_id"]].merge(
    model_data[["order_id"]],
    on="order_id",
    how="outer",
    validate="one_to_one",
    indicator=True,
)
orders_target_unmatched = int((orders_target["_merge"] != "both").sum())

# 4) orders → customers: many_to_one
orders_customers = orders[["order_id", "customer_id"]].merge(
    customers[["customer_id"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched_customers = int((orders_customers["_merge"] != "both").sum())

# 5) order_items → orders: 상세의 FK 연결 확인
items_orders = items[["order_id"]].merge(
    orders[["order_id"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched_order_items = int((items_orders["_merge"] != "both").sum())

checks = pd.DataFrame([
    ["line_total 불일치 행", line_mismatch_count,
     "PASS" if line_mismatch_count == 0 else "FAIL"],
    ["order_total 불일치/누락 주문", target_mismatch_count,
     "PASS" if target_mismatch_count == 0 else "FAIL"],
    ["orders ↔ target 미매칭 행", orders_target_unmatched,
     "PASS" if orders_target_unmatched == 0 else "FAIL"],
    ["주문-고객 미매칭 행", unmatched_customers,
     "PASS" if unmatched_customers == 0 else "FAIL"],
    ["상세-주문 FK 미매칭 행", unmatched_order_items,
     "PASS" if unmatched_order_items == 0 else "FAIL"],
], columns=["check", "value", "status"])

display(checks)
assert checks["status"].eq("PASS").all(), "FAIL 항목을 확인하세요."
print("Target 계산 및 관계 검증: PASS")

,check,value,status
0,line_total 불일치 행,0,PASS
1,order_total 불일치/누락 주문,0,PASS
2,orders ↔ target 미매칭 행,0,PASS
3,주문-고객 미매칭 행,0,PASS
4,상세-주문 FK 미매칭 행,0,PASS


Target 계산 및 관계 검증: PASS


## 4. Feature Audit을 확인합니다

In [25]:
feature_audit = build_feature_audit()
display(feature_audit)

,column,selected,role,reason
0,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,avg_unit_price,False,forbidden,주문 상세에서 만든 목표 대리 변수
7,customer_id,False,forbidden,고객 식별자
8,item_count,False,forbidden,주문 상세가 확인된 뒤 계산되는 사후 집계
9,line_total,False,forbidden,목표값을 구성하는 주문 상세 금액


## 5. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [26]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

,split,rows,ratio_pct,start_date,end_date
0,train,244,81.33,2025-07-09,2026-04-13
1,test,56,18.67,2026-04-14,2026-07-08


Chronological split gate: PASS


In [27]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

Train: (244, 6) Final Test: (56, 6)


## 6. Pipeline 후보 모델을 준비합니다

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [28]:
models = make_regression_models(random_state=42)
print(list(models))

['Baseline Mean', 'Linear Regression', 'Random Forest']


## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [29]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,423004.808743,44153.043466,-0.022821
1,Random Forest,5,437352.803929,26318.793429,-0.109687
2,Linear Regression,5,461386.286905,77792.799324,-0.310517


## 8. 비베이스라인 모델 선택을 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [30]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

Frozen selected model: Random Forest


## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [31]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.699620,-0.001121,0.00
1,Random Forest,selected_by_train_cv,326438.478132,458819.917234,548190.446723,-0.103298,-3.47


### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

## 10. 실제값·예측값·잔차를 진단합니다

In [32]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')

,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-06-15,1877000,605528.715709,1.271471e+06,1.271471e+06,Random Forest
1,21,2026-06-19,1767000,756889.654706,1.010110e+06,1.010110e+06,Random Forest
2,145,2026-05-27,1863000,861784.708271,1.001215e+06,1.001215e+06,Random Forest
3,96,2026-05-11,1690000,694946.179725,9.950538e+05,9.950538e+05,Random Forest
4,183,2026-05-23,1685000,745836.316118,9.391637e+05,9.391637e+05,Random Forest
5,168,2026-06-11,10000,897415.394314,-8.874154e+05,8.874154e+05,Random Forest
6,15,2026-05-23,1677000,849179.542426,8.278205e+05,8.278205e+05,Random Forest
7,179,2026-07-08,86000,904745.854532,-8.187459e+05,8.187459e+05,Random Forest
8,282,2026-04-24,1566000,758941.292777,8.070587e+05,8.070587e+05,Random Forest
9,73,2026-06-14,72000,871636.144070,-7.996361e+05,7.996361e+05,Random Forest


주의: order_id가 포함된 이 표는 내부 진단용입니다.


## 11. 자동 Validation Evidence를 만듭니다

In [33]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다

In [34]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

model_data_internal -> C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch09_regression_model_data_internal.csv
split_summary -> C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch09_regression_split_summary.csv
feature_audit -> C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch09_regression_feature_audit.csv
cv_summary -> C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch09_regression_cv_summary.csv
model_comparison -> C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch09_regression_model_comparison.csv
predictions_internal -> C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch09_regression_predictions_internal.csv
validation -> C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch09_regression_validation.csv
checklist -> C:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch09_regression_checklist.csv
report -> C:\Users\ojh23\Documents\projects\

## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [15]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

Selected by Train CV: Random Forest


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 14. 해석

낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.

LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다.

## 15. 완료 체크

- [x] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [x] Target과 예측 시점을 설명할 수 있다.
- [x] 금지 feature가 입력에 없음을 확인했다.
- [x] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [x] 전처리는 Pipeline 내부에서 학습된다.
- [x] Train TimeSeriesSplit으로 후보를 비교했다.
- [x] Final Test 전에 Selected Model을 고정했다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [x] MAE, RMSE, R²를 함께 읽었다.
- [x] 내부 식별자 결과와 공개 보고서를 구분했다.
- [x] Validation Evidence가 모두 PASS다.

# Chapter 09 답안. 회귀 분석으로 숫자 예측하기

## 제출 정보
- 이름: 오병희
- GitHub ID: OBHAR
- 작성일: 2026-11-05
- 최종 Notebook URL: https://github.com/OBHAR/llm-data-analysis-study/blob/main/chapter09/chapter09.ipynb

## 1. 예측 문제 정의
- **Target:** 주문 1건의 상세 금액을 합산한 `order_total`
- **예측 단위:** 주문 1건
- **예측 시점:** 주문 날짜·결제 수단과 고객 기본 정보는 확인되지만 주문 상세 품목, 수량, 단가가 정해지기 전으로 가정
- **예측 시점에 알 수 있는 정보:** `order_month`, `order_dayofweek`, `age`, `payment_method`, `gender`, `city`
- **예측 시점에 알 수 없는 정보:** 주문 상세의 `quantity`, `unit_price`, `line_total`, 상세 집계값, `order_total`; 사후 확정될 수 있는 `order_status`; 식별자

### 나의 해석과 판단
주문 금액을 알기 전에 예측하는 문제를 만들기 위해 이 시점을 정했다. 주문 상세값을 Feature로 쓰면 Target을 직접 또는 간접적으로 알려 주므로 제외했다. 결제 수단과 고객 정보가 실제 업무에서도 예측 시점에 조회 가능한지는 추가 확인이 필요하다.

## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
|---|---|---|---|---|---|
| `order_month`, `order_dayofweek` | 예 | 아니오 | 아니오 | 예 | 주문 날짜에서 계산 가능 |
| `age`, `gender`, `city` | 가정상 예 | 아니오 | 아니오 | 예 | 고객 기본 정보로 가정. 실제 접근 가능 시점 확인 필요 |
| `payment_method` | 가정상 예 | 아니오 | 아니오 | 예 | 예측 시점에 결제 수단이 선택되어 있다는 전제 |
| `order_total` | 아니오 | 예측 대상 | 아니오 | 아니오 | Target 자체 |
| `line_total` | 아니오 | 예 | 아니오 | 아니오 | Target을 구성하는 상세 금액 |
| `quantity`, `unit_price` | 아니오 | 예 | 아니오 | 아니오 | `line_total` 계산 재료 |
| `item_count`, `total_quantity`, `avg_unit_price` | 아니오 | 간접 대리 변수 | 사후 집계 | 아니오 | 주문 상세 확인 뒤 생성됨 |
| `order_status` | 보장 불가 | 아니오 | 사후 정보 | 아니오 | 예측 이후 확정될 수 있음 |
| `order_id`, `customer_id` | 아니오 | 아니오 | 식별자 | 아니오 | 일반화 신호가 아니며 식별자 노출 위험도 있음 |
| `product_id` | 아니오 | 아니오 | 상세 기반 식별자 | 아니오 | 주문 상세가 확인되어야 알 수 있음 |

- **금지 Feature overlap 개수:** 0 (Validation PASS)
- **가장 위험한 Feature:** `order_total`, `line_total`
- **이유:** `order_total`은 Target 그 자체이고, `line_total`은 Target을 직접 구성한다. 둘 중 하나를 사용하면 누수로 성능이 부당하게 높아질 수 있다.

![Feature Leakage Audit 캡처](images/step02_leakage.png)

## 3. Target 생성과 관계 검증

### 계산 기준
```text
line_total = quantity × unit_price
order_total = 같은 order_id의 line_total 합계
```

- **데이터 크기:** 고객 150행, 주문 300행, 주문 상세 764행. 주문 단위 모델 데이터 300행 생성
- **line_total 검증 결과:** 산식 불일치 0행, `PASS`.
- **orders ↔ target 관계:** 미매칭 0행, `one_to_one` 검증 통과.
- **orders → customers 관계:** 미매칭 0행, `many_to_one` 검증 통과. 주문 상세 → 주문 FK도 미매칭 0행이다.

### 나의 해석과 판단
관계 검증 없이 병합하면 주문 상세가 여러 개인 주문이 중복되어 주문이 과대 집계될 수 있고, FK 불일치나 중복 고객 때문에 행이 빠지거나 늘어날 수 있다. 이번 실행에서는 산식 불일치, Target 누락·미매칭, 고객 FK 미매칭, 주문 상세 FK 미매칭이 모두 0행으로 확인됐다. pandas merge의 `validate` 검사도 오류 없이 통과했다.

![Target 계산 및 관계 검증 결과](images/step03_target_relationship_validation.png)

## 4. Train / Final Test 분할

- **Train 시작일:** 2025-07-09
- **Train 종료일:** 2026-04-13
- **Train 행 수:** 244
- **Final Test 시작일:** 2026-04-14
- **Final Test 종료일:** 2026-07-08
- **Final Test 행 수:** 56
- **같은 달력 날짜 중복 여부:** 없음. `train max date < final test min date` 확인 및 Chronological split gate PASS

### 분할 판단
미래 주문을 과거 자료로 예측하는 상황을 평가하기 위해 시간 순서로 나눴다. 무작위 분할은 미래 자료가 학습에 섞여 실제보다 낙관적인 평가를 만들 수 있다. Final Test는 모델 선택에 사용하지 않았다.

![시간 분할 캡처](images/step04_split.png)

## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
|---|---:|---:|---:|---|
| Baseline Mean | 423,004.81 | 44,153.04 | -0.0228 | 비교 기준 |
| Linear Regression | 461,386.29 | 77,792.80 | -0.3105 | 후보 |
| Random Forest | 437,352.80 | 26,318.79 | -0.1097 | 선택된 비베이스라인 모델 |

### Train CV로 선택한 비베이스라인 모델
- **Selected Model:** Random Forest
- **선택 기준:** Linear Regression과 Random Forest 중 Train CV 평균 MAE가 더 낮음
- **Final Test 결과를 보기 전에 선택했는가:** 예

### 나의 해석과 판단
CV 평균은 전반적인 오차 수준을, 표준편차는 시간 구간에 따라 성능이 얼마나 달라지는지를 보여 준다. Random Forest 표준편차는 두 후보 중 낮았지만 평균 MAE는 Baseline보다 높고 CV R² 평균도 음수여서 우수한 예측력을 입증하지는 못했다.

## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
|---|---|---:|---:|---:|---:|---:|
| Baseline Mean | baseline | 416,266.19 | 443,449.65 | 522,189.70 | -0.0011 | 0.00% |
| Random Forest | selected_by_train_cv | 326,438.48 | 458,819.92 | 548,190.45 | -0.1033 | -3.47% |

![Final Test 모델 비교 캡처](images/step06_final_metrics.png)

### 결과 관찰
Final Test에서 고정 Random Forest의 MAE는 Baseline보다 15,370.27 높았다. 개선율이 -3.47%이므로 복잡한 모델이 기준 모델보다 나은 결과를 보이지 않았다.

### 나의 해석과 판단
- **Baseline보다 개선되었는가?** 아니다. MAE가 3.47% 악화됐다.
- **MAE와 RMSE 차이:** Random Forest RMSE가 MAE보다 높아 큰 오차가 평가에 더 큰 영향을 준다.
- **R²:** Random Forest의 -0.1033은 이 테스트 구간에서 평균 기준 예측보다 설명력이 낮았음을 뜻한다. 음수 R²만으로 코드 오류라고 판단하지 않는다.

### 업무·분석적 의미
Random Forest의 평균 절대 오차는 약 458,820 데이터 금액 단위다. 현재 결과로는 주문 금액 예측을 업무 판단에 바로 활용하기 어렵다. 데이터 단위가 원이라면 이 값은 약 458,820원이다.

## 7. 대표 오차 사례

잔차는 `실제값 - 예측값`으로 계산했다. 아래는 절대 오차가 큰 상위 사례이며, 설명은 원인 단정이 아니라 확인할 가설이다.

### 사례 1
- 실제값: 1,877,000
- 예측값: 605,528.72
- residual: 1,271,471.28
- absolute error: 1,271,471.28
- 관찰: 실제 합계를 크게 낮게 예측했다.
- 가능한 이유 후보: 현재 Feature가 상품 구성이나 주문 상세 규모를 표현하지 못했을 가능성이 있다.
- 추가 확인: 예측 시점에 이용 가능한 상품·카테고리 정보가 있는지, 고액 주문의 구성과 데이터 품질을 점검한다.

### 사례 2
- 실제값: 1,767,000
- 예측값: 756,889.65
- residual: 1,010,110.35
- absolute error: 1,010,110.35
- 관찰: 실제 합계를 약 101만 단위 낮게 예측했다.
- 가능한 이유 후보: 제한된 주문·고객 특성만으로 고액 주문을 구분하지 못했을 수 있다.
- 추가 확인: 상품 수·가격 분포 등 예측 시점에 실제 확보 가능한 데이터와 더 긴 기간의 오차 패턴을 확인한다.

현재 자료만으로 오차 원인을 확정할 수 없다. 내부 진단 출력에 식별자가 포함되므로 공개 답안에는 주문 ID를 적지 않았다.

## 8. Validation Evidence

| check | value | status |
|---|---:|---|
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

### FAIL 항목
- **없음.** 저장된 Notebook Validation 여섯 항목은 모두 PASS다.

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
`order_id`가 포함된 예측 결과는 개별 주문과 금액을 연결할 수 있으므로 Internal 파일로 관리한다. 제출 답안에는 모델 성능의 집계 지표와 익명화한 오차 사례만 제시하고, 식별자를 포함한 내부 파일은 공개하지 않는다.

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거
1. Train CV에서 Random Forest CV MAE 평균은 437,352.80으로 Baseline의 423,004.81보다 높았다. CV R² 평균도 -0.1097이다.
2. Final Test에서 MAE 458,819.92, RMSE 548,190.45, R² -0.1033으로 Baseline보다 MAE가 3.47% 악화됐다.
3. 금지 Feature overlap은 0이고 시간순 분할 및 Validation 여섯 항목은 PASS다. 따라서 이 실행에서 확인된 주요 문제는 누수·분할 실패가 아니라 허용 Feature의 낮은 예측력이다.
4. 상위 오차 사례에서 고액 주문을 크게 낮게 예측했지만 데이터에 없는 원인을 단정할 수 없다.

### 업무적 위험
예측값을 주문 금액이나 자원 배분의 확정 근거로 사용하면 고액 주문을 과소 예측할 수 있다. Baseline보다 나은 성능을 입증하지 못했으므로 현재는 참고 지표로도 제한적으로 다뤄야 한다.

### 현재 데이터의 한계
주문별 300건, 시간 분할 후 Final Test 56건이며, 허용 Feature에는 상품 구성·수량·단가 등 금액 변동을 설명할 수 있는 정보가 들어 있지 않다. 해당 정보는 예측 시점에 사용 가능한지 먼저 검토해야 한다. 프로모션 등 데이터에 없는 원인도 이 결과로 판단할 수 없다.

### 다음 개선 우선순위
1. 각 Feature가 실제로 예측 시점에 확보되는지 업무 흐름에서 확인한다.
2. 예측 시점에 허용되는 추가 변수를 확보하고, 누수 점검 후 Train CV로 다시 평가한다.
3. 더 긴 기간과 별도의 미래 구간에서 Baseline 대비 개선이 반복되는지 검증한다.

## 11. 전체 재실행 확인

실행 명령:
```powershell
python scripts/prepare_ch09_data.py
python scripts/run_regression_analysis.py
```

- **실행 성공 여부:** `prepare_ch09_data.py`와 `run_regression_analysis.py` 두 스크립트를 재실행했고, 성공 메시지 및 결과 파일 저장 경로가 출력됐다. Validation 6개 항목도 모두 PASS했다.
- **Train CV Selected Model:** Random Forest
- **Final Validation 전체 PASS 여부:** 예, 6/6 PASS

![두 스크립트 실행 및 Validation 결과](images/step11_rerun_validation.png)
- **Notebook 결과와 스크립트 결과의 일관성:** 스크립트 실행 후 저장된 Validation에서 선택 모델 존재 여부, Baseline 및 Frozen Model 포함 여부가 PASS이며, 6개 검사가 모두 통과했다.

## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산 및 병합 관계 상세 검증 5개 항목이 모두 0행·PASS임을 확인했습니다.
- [x] Train과 Final Test를 시간순으로 엄격히 분리했습니다.
- [x] 전처리를 Pipeline 내부에서 Train으로만 학습했습니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 나누어 정리했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능을 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다. (GitHub 업로드 후 입력)